Machine translation (MT) is the automated process of using software to translate text or speech from one natural language to another without human involvement.

We’ll be working with an English-to-Spanish translation dataset available at
www.manythings.org/anki/.

In [3]:
#Downloading the dataset
!wget https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
!unzip -q spa-eng.zip

--2026-03-26 13:54:10--  https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.202.207, 64.233.181.207, 74.125.69.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.202.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2638744 (2.5M) [application/zip]
Saving to: ‘spa-eng.zip’

spa-eng.zip         100%[===================>]   2.52M  --.-KB/s    in 0.01s   

2026-03-26 13:54:10 (205 MB/s) - ‘spa-eng.zip’ saved [2638744/2638744]



The text file contains one example per line: an English sentence, followed by a tab
character, followed by the corresponding Spanish sentence. Let’s parse this file.

## Getting data ready:

In [11]:
text_file = '/kaggle/working/spa-eng/spa.txt'
with open(text_file) as f:
    lines = f.read().split('\n') [:-1]
text_pairs = []

for line in lines:
    english,spanish = line.split('\t')
    spanish = "[start] " + spanish + " [end]" 
    text_pairs.append((english,spanish))

[start] and [end] are special tokens, often called start token and end token

This helps the model understand:
* Where the sentence starts
* Where the sentence ends
* When to stop generating output

Lets look at one of our text_pairs:

In [12]:
import random 
print(random.choice(text_pairs))

("He doesn't remember me anymore.", '[start] Él ya no me recuerda. [end]')


In [13]:
print('total number of samples: ',len(text_pairs))

total number of samples:  118964


In [14]:
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples

train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_samples = text_pairs = text_pairs[num_train_samples + num_val_samples:]

## Vectorizing the English and Spanish text pairs:
let’s prepare two separate TextVectorization layers: one for English and one
for Spanish. We’re going to need to customize the way strings are preprocessed:
* We need to preserve the "[start]" and "[end]" tokens that we’ve inserted. By
default, the characters [ and ] would be stripped, but we want to keep them
around so we can tell apart the word “start” and the start token "[start]".
* Punctuation is different from language to language! In the Spanish Text
Vectorization layer, if we’re going to strip punctuation characters, we need to
also strip the character ¿.

Note that for a non-toy translation model, we would treat punctuation characters as sep
arate tokens rather than stripping them. In our case, for simplicity, we’ll get rid of all punctuation.

In [ ]:
import tensorflow as tf
import string 
import re

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[","")
strip_chars = strip_chars.replace("]","")

def custom_standardization(input_string):
'''Prepare a custom string standardization function for the Spanish TextVectorization layer: 
it preserves [ and ] but strips ¿ (as well as all other characters 
from strings.punctuation)'''
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(
        lowercase, f"[{re.escape(strip_chars)}]", "")

vocab_size = 15000 #to keep it simple
sequence_length = 20 #to keep it simple

#the English layer
source_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = vocab_size,
    output_mode = 'int',
    output_sequence_length = sequence_length,
)

#the Spanish layer
target_vectorizer = tf.keras.layers.TextVectorization(
    max_tokens = vocab_size,
    output_mode = 'int',
    output_sequence_length = sequence_length + 1,
    standardize = custom_standardization,
)

train_english_texts = [pair[0] for pair in text_pairs]
train_spanish_texts = [pair[1] for pair in text_pairs]

source_vectorizer.adapt(train_english_texts)
target_vectorizer.adapt(train_spanish_texts)